# Using Whisper for Video Transcription in FiftyOne

[Whisper](https://huggingface.co/openai/whisper-large-v3) is OpenAI's speech-to-text model. Wrapped as a [remotely-sourced zoo model](https://docs.voxel51.com/model_zoo/remote.html), it attaches timestamped transcripts to a video dataset with a single `dataset.apply_model(...)` call.

In a single pass the model writes **two** sample fields:

- a `fo.TemporalDetections` of speech segments — frame-aligned, scrubbable in the App, each carrying a `text` attribute
- a flat transcript **string** field — queryable with text filters, embeddable, usable as VLM context

**What you'll learn:**

1. Load a video dataset from the Hugging Face Hub
2. Register a remote zoo model source and load Whisper from it
3. Transcribe videos and map both outputs to fields with `label_field`
4. Explore segments in the App and query transcripts with `ViewField`
5. Change runtime parameters (language, chunking) without reloading weights

**Requirements:** a CUDA/MPS GPU is recommended for `large-v3`, and `ffmpeg` must be on your PATH (the pipeline uses it to demux audio from the videos).

In [ ]:
!pip install -q -U fiftyone torch "transformers>=4.56" accelerate umap-learn einops

## Load a video dataset

We'll use [`Voxel51/action100m_tiny_subset`](https://huggingface.co/datasets/Voxel51/action100m_tiny_subset) — instructional videos (90s clips at 480p) with plenty of speech, so there's something for Whisper to transcribe. `max_samples=10` keeps the download and transcription time small; raise it if you want more.

One thing to notice: this dataset already ships with its **own** ASR annotations from the Action100M pipeline (`transcript` and `transcript_segments` fields). We'll write Whisper's output to separate `whisper_*` fields so we can compare the two later instead of overwriting the originals.

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz
from fiftyone import ViewField as F
from fiftyone.utils.huggingface import load_from_hub

dataset = load_from_hub("Voxel51/action100m_tiny_subset", max_samples=10)

print(dataset)

## Explore the videos in the App

Launch the FiftyOne App to get a feel for the data before running the model. Click into a sample and play it — these are how-to videos with a narrator, which is exactly the kind of audio Whisper handles well.

In [ ]:
session = fo.launch_app(dataset)

## Register the remote zoo model source

FiftyOne's model zoo can load models from [remote sources](https://docs.voxel51.com/model_zoo/remote.html) — GitHub repos that ship a `manifest.json` plus the model implementation. Registering the source once makes its models available to `foz.load_zoo_model()` by name, just like built-in zoo models.

In [ ]:
foz.register_zoo_model_source(
    "https://github.com/harpreetsahota204/whisperx",
    overwrite=True,
)

## Load Whisper

Three model sizes are available — `whisper-large-v3` (best accuracy), `whisper-large-v3-turbo` (faster, small accuracy cost), and `whisper-large-v2`.

All constructor arguments have sensible defaults:

- `device=None` auto-selects `cuda > mps > cpu`
- `torch_dtype=None` picks `float16` on cuda/mps and `float32` on cpu
- `chunk_length_s=None` uses Whisper's native sequential long-form decoding (most accurate); set e.g. `30` for the faster chunked algorithm
- `language=None` auto-detects the spoken language per sample

The weights load once here and are reused for every sample.

In [ ]:
model = foz.load_zoo_model(
    "whisper-large-v3",  # or: whisper-large-v3-turbo, whisper-large-v2
    device=None,         # None auto-selects cuda > mps > cpu
    torch_dtype=None,    # None -> float16 on cuda/mps, float32 on cpu
    batch_size=16,
    language=None,       # e.g. "en" to force; None auto-detects per sample
)

## Compute metadata (required)

Whisper returns timestamps in **seconds**, but FiftyOne's `TemporalDetection` stores a frame-number `support`. The conversion uses each video's frame rate from `sample.metadata`, so metadata must be populated before applying the model.

In [ ]:
dataset.compute_metadata()

## Transcribe the videos

`predict()` returns a dict with two outputs per sample — `{"segments": TemporalDetections, "transcript": str}` — and `label_field` controls where they land:

| `label_field` | Resulting fields |
| --- | --- |
| `{"segments": "whisper_segments", "transcript": "whisper_transcript"}` | exact names |
| `"audio"` (string) | prefixed: `audio_segments`, `audio_transcript` |
| `None` | the dict keys verbatim: `segments`, `transcript` |

We use the dict form to keep Whisper's output clearly separated from the dataset's bundled `transcript*` fields. This is the long-running cell — each video gets transcribed by the model here.

In [ ]:
dataset.apply_model(
    model,
    label_field={"segments": "whisper_segments", "transcript": "whisper_transcript"},
)

## Inspect the results

Each sample now has both fields. The flat `whisper_transcript` is a plain string, and `whisper_segments` holds one `TemporalDetection` per Whisper segment — `support` is the `[first, last]` frame range, and `text` carries the words spoken in that span.

In [ ]:
sample = dataset.first()

# Flat transcript: a top-level StringField, queryable on its own
print(sample.whisper_transcript[:500])
print()

# Timestamped segments: TemporalDetections with text per segment
for det in sample.whisper_segments.detections[:5]:
    print(det.support, det.text)

## Scrub the segments in the App

Refresh the App and open a sample — the `whisper_segments` field appears on the video timeline. As you scrub or play the video, the active speech segment highlights, and clicking a segment shows its `text` attribute. This is the payoff of frame-aligned `TemporalDetections` over a plain string.

In [ ]:
session.refresh()

## Compare against the dataset's bundled ASR

Because we wrote Whisper's output to separate fields, the original Action100M ASR annotations are still intact. Comparing the two side by side is a quick sanity check on transcription quality — and a nice example of why keeping model outputs in their own fields beats overwriting.

In [ ]:
sample = dataset.first()

print("=== Original Action100M ASR ===")
print((sample.transcript or "")[:300])
print()
print("=== Whisper large-v3 ===")
print((sample.whisper_transcript or "")[:300])

## Query the transcripts

This is where the flat string field earns its keep. `ViewField` expressions let you filter samples by what was *said* in them — and `filter_labels` narrows the segments themselves, so the App shows only the matching spans on the timeline.

In [ ]:
keyword = "water"  # try any word you spotted in the transcripts

# Samples whose transcript mentions the keyword
spoken = dataset.match(
    F("whisper_transcript").contains_str(keyword, case_sensitive=False)
)
print(f"{len(spoken)} of {len(dataset)} videos mention '{keyword}'")

# Narrow to just the segments where it is said, and show them in the App
segments = spoken.filter_labels(
    "whisper_segments", F("text").contains_str(keyword, case_sensitive=False)
)
session.view = segments

## Tune runtime parameters — no reload needed

`batch_size`, `chunk_length_s`, and `language` are exposed as **setters** on the model, so you can change them between `apply_model` calls without reconstructing the model (which would reload multi-GB weights).

Here we force English, switch to the faster chunked long-form algorithm, and re-transcribe two samples. Passing a plain **string** as `label_field` demonstrates the prefix behavior — outputs land in `fast_segments` / `fast_transcript`.

In [ ]:
model.language = "en"        # skip per-sample language detection
model.chunk_length_s = 30    # faster chunked long-form decoding
model.batch_size = 8

two_videos = dataset.take(2, seed=51)
two_videos.apply_model(model, label_field="fast")

print(two_videos.first().fast_transcript[:300])

## Embed the transcripts

The flat transcript field has one more trick: it's plain text, so we can run it through a **text embedding model** and treat spoken content like any other vector space — cluster videos by what's *said* in them, find outliers, or do semantic (rather than keyword) search.

We'll use [`jinaai/jina-embeddings-v3`](https://huggingface.co/jinaai/jina-embeddings-v3) and store one embedding per sample. Samples that were skipped during transcription (e.g. no audio) have no transcript, so we skip them here too.

In [ ]:
import os
from transformers import AutoModel

# quiet a tokenizers warning (transformers/tokenizers, not FiftyOne)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

embedder = AutoModel.from_pretrained(
    "jinaai/jina-embeddings-v3",
    trust_remote_code=True,
    device_map=model.device,  # reuse the device Whisper picked
)

for sample in dataset.iter_samples(autosave=True, progress=True):
    if not sample["whisper_transcript"]:
        continue

    embeddings = embedder.encode(
        [sample["whisper_transcript"]],  # model expects a list of strings
        task="separation",               # embeddings tuned for clustering
    )
    sample["transcript_embeddings"] = embeddings.squeeze()

## Visualize the embeddings with UMAP

The [FiftyOne Brain](https://docs.voxel51.com/brain.html#visualizing-embeddings) reduces the embeddings to 2D with UMAP so they can be explored in the App's **Embeddings panel** — videos that talk about similar things land near each other, regardless of what they look like. `create_index=True` lets you lasso points in the panel to select the corresponding samples.

> Requires the `umap-learn` package (included in the install cell at the top).

In [ ]:
import fiftyone.brain as fob

results = fob.compute_visualization(
    dataset,
    embeddings="transcript_embeddings",
    method="umap",
    brain_key="transcript_viz",
    num_dims=2,
    create_index=True,
    skip_failures=True,
)

# Open the Embeddings panel in the App and select "transcript_viz"
session.refresh()

## Summary

You transcribed a video dataset with one `apply_model` call and got two complementary fields per sample:

- **`whisper_segments`** (`TemporalDetections`) — frame-aligned speech spans with `text`, scrubbable on the App timeline
- **`whisper_transcript`** (string) — the full transcript, queryable with `ViewField` expressions

Along the way you saw how `label_field` maps the model's dict output to fields (exact dict / string prefix / verbatim keys), how runtime setters let you retune `language`, `chunk_length_s`, and `batch_size` without reloading weights, and how text embeddings of the transcripts open up UMAP exploration of spoken content in the Embeddings panel.

**Where to go next:**

- Build a similarity index over `transcript_embeddings` for semantic search over *spoken* content ([FiftyOne Brain similarity](https://docs.voxel51.com/brain.html#similarity))
- Feed transcripts to a VLM as context for caption or QA workflows
- Try `whisper-large-v3-turbo` for higher throughput on larger datasets

**Notes:**

- Videos with no audio stream fail per-sample; `apply_model` skips them with a warning by default (`skip_failures=False` to raise instead)
- GPU memory is freed automatically when `apply_model` finishes